In [28]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# /kaggle/input/ 是只读目录，无法修改或删除文件。
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# /kaggle/working/（工作目录）
# 用途：存储你的代码、输出文件（如模型、图表）、临时数据。
# 配额：
# 默认限制：20GB（用户可写入空间）。
# 保存规则：只有通过 "Save & Run All" 保存的版本会持久化，其他临时修改在 Kernel 重启后丢失。
# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [29]:
import os
stat = os.statvfs('/kaggle/working/')
free_space_gb = stat.f_bavail * stat.f_frsize / (1024 ** 3)
print(f"剩余空间: {free_space_gb:.2f} GB")

剩余空间: 12.50 GB


1，使用模型来对图片进行描述

数据集位置:/kaggle/input/fruit-and-vegetable-image-recognition

模型位置:/kaggle/input/qwen2.5/transformers/1.5b-instruct/1

In [30]:
import torch

def check_gpu_availability():
    # 1. 检查 PyTorch 是否能检测到 GPU
    if torch.cuda.is_available():
        print("✅ GPU 可用！")
        
        # 2. 显示 GPU 信息
        print(f"🖥️ 当前使用的 GPU: {torch.cuda.get_device_name(0)}")
        print(f"⚡ CUDA 版本: {torch.version.cuda}")
        print(f"🧠 总显存: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
        print(f"📦 当前显存占用: {torch.cuda.memory_allocated() / (1024**2):.2f} MB")
        print(f"📊 计算能力: {torch.cuda.get_device_properties(0).major}.{torch.cuda.get_device_properties(0).minor}")
    else:
        print("❌ GPU 不可用，将使用 CPU。")
        print("💡 提示：如果你期望使用 GPU，请检查：")
        print("   - 是否安装了 CUDA 版本的 PyTorch？")
        print("   - 是否安装了正确的 GPU 驱动？")
        print("   - 是否在支持 GPU 的环境中运行（如 Colab、Kaggle、本地 GPU 机器）？")

if __name__ == "__main__":
    check_gpu_availability()

✅ GPU 可用！
🖥️ 当前使用的 GPU: Tesla P100-PCIE-16GB
⚡ CUDA 版本: 12.4
🧠 总显存: 15.89 GB
📦 当前显存占用: 7175.04 MB
📊 计算能力: 6.0


加载Qwen2.5的模块

In [31]:
# 
# import warning
from transformers import AutoModelForCausalLM, AutoTokenizer,AutoProcessor
from transformers import VisionEncoderDecoderModel, ViTFeatureExtractor
import torch

# 模型路径（根据你的实际路径调整）
model_path = "/kaggle/input/qwen2.5/transformers/1.5b-instruct/1"

# 加载模型和处理器
try:
    # processor = AutoProcessor.from_pretrained(model_path)
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.float16,  # 半精度减少显存占用
        device_map="auto",          # 自动分配设备（GPU/CPU）
        trust_remote_code=True      # 允许运行模型自定义代码（Qwen需要）
    )
    model = model.eval()  # 设置为推理模式
    print("✅ 模型加载成功！")
except Exception as e:
    print(f"❌ 模型加载失败: {e}")


✅ 模型加载成功！


In [32]:
# 列出 Notebook 磁盘（/kaggle/working/）
import os
import shutil
print("Notebook 磁盘内容:")
for item in os.listdir('/kaggle/working/'):
    print(item)

# 列出 Dataset 磁盘（/kaggle/input/）
print("\nDataset 磁盘内容:")
for item in os.listdir('/kaggle/input/'):
    print(item)
print("------------------------")
total, used, free = shutil.disk_usage("/kaggle/input/")

print(f"总空间: {total // (2**30)} GB")
print(f"已用空间: {used // (2**30)} GB")
print(f"剩余空间: {free // (2**30)} GB")


Notebook 磁盘内容:
qwen2.5-vl-3b
.virtual_documents

Dataset 磁盘内容:
rvf10k
qwen2.5-vl
qwen2.5
fruit-and-vegetable-image-recognition
------------------------
总空间: 19 GB
已用空间: 7 GB
剩余空间: 12 GB


In [33]:
# 测试推理
if "model" in locals():  # 确保模型已加载
    input_text = "Hello, how are you?"
    inputs = tokenizer(input_text, return_tensors="pt").to("cuda" if torch.cuda.is_available() else "cpu")
    outputs = model.generate(**inputs, max_new_tokens=50)
    print("模型输出:", tokenizer.decode(outputs[0], skip_special_tokens=True))

模型输出: Hello, how are you? I'm a bit concerned about my mental health. How can I help myself feel better?

As an AI language model, I am not capable of providing professional medical advice or diagnosis. However, there are several steps you can take to improve your mental health


加载Qwen2.5 VL的模块

In [34]:
# !pip install qwen_vl_utils

In [35]:
# !cp -r /kaggle/input/qwen2.5-vl/transformers/3b-instruct/2 ./qwen2.5-vl-3b
print("finished")
# 复制文件

finished


In [36]:
# ！rm -rf /kaggle/working/*
# 删除所有working文件

In [37]:
import json

config_data = {
  "min_pixels": 3136,
  "max_pixels": 12845056,
  "patch_size": 14,
  "temporal_patch_size": 2,
  "merge_size": 2,
  "image_mean": [
    0.48145466,
    0.4578275,
    0.40821073
  ],
  "image_std": [
    0.26862954,
    0.26130258,
    0.27577711
  ],
  "image_processor_type": "Qwen2VLImageProcessor",
  "processor_class": "Qwen2_5_VLProcessor"
}
# note:"image_processor_type": "Qwen2VLImageProcessor",这里的问题，原来这里是Qwen2_5_VLImageProcessor
output_filename = "qwen2.5-vl-3b/preprocessor_config.json"
# /kaggle/working/2/preprocessor_config.json
with open(output_filename, 'w', encoding='utf-8') as f:
    json.dump(config_data, f, ensure_ascii=False, indent=4)

print("finished")

finished


In [38]:
# from transformers import Qwen2_5_VLImageProcessor
# note 因为这个不能导入，所以原来的json文件不行,需要重写

In [39]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor,AutoTokenizer
from qwen_vl_utils import process_vision_info
import torch

# model_path = "/kaggle/input/qwen2.5-vl/transformers/3b-instruct/2"
# /kaggle/input/qwen2.5-vl/transformers/3b-instruct/2
model_path = "/kaggle/working/qwen2.5-vl-3b"
try:
    # 仅加载分词器和模型
    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    processor = AutoProcessor.from_pretrained(model_path)
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        model_path, 
        torch_dtype=torch.float16, 
        device_map="auto"
    )
    
    model.eval()
    print("✅ 模型加载成功")
except Exception as e:
    print(f"❌ 加载失败: {e}")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

✅ 模型加载成功


In [40]:
from PIL import Image
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
import torch

# 读取图片
image_path = "/kaggle/input/rvf10k/rvf10k/train/real/00000.jpg"
image = Image.open(image_path)
display(image)
print(f"✅ 图片加载成功: {image.size}")

# 模型路径（使用你之前的设置）
model_path = "/kaggle/working/qwen2.5-vl-3b"

# 加载处理器和模型
processor = AutoProcessor.from_pretrained(model_path)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
model.eval()

# 构建对话
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": "Describe this image in detail."}
        ]
    }
]

# 处理输入
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = processor(
    text=[text],
    images=[image],
    padding=True,
    return_tensors="pt"
)
inputs = inputs.to(model.device)

# 生成回答
with torch.no_grad():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False
    )

# 解码结果
generated_ids_trimmed = generated_ids[:, inputs["input_ids"].shape[1]:]
output = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True)[0]

print("\n🤖 模型描述:")
print(output)

✅ 图片加载成功: (256, 256)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



🤖 模型描述:
The image shows a baby lying down, possibly in a crib or a similar setting. The baby is wearing a light green and white striped shirt. The background appears to be a soft, blue fabric, which could be part of the baby's bedding or a cushion. The baby has short hair and is looking slightly to the side with a calm expression on their face.
